In [ ]:
# creates the price_history.parquet from the .db file we downloaded
# import duckdb

# con = duckdb.connect()
# con.execute("SET memory_limit='3GB'")
# con.execute("SET threads=16")
# con.execute("PRAGMA temp_directory='/content/duckdb_tmp'")  # lets it spill to SSD instead of RAM when needed

# con.execute("INSTALL sqlite; LOAD sqlite;")
# con.execute("ATTACH '../raw/price_history_backup.db' AS pm (TYPE sqlite)")

# con.execute("SELECT COUNT(*) FROM pm.price_history").df()

In [ ]:
# con.execute("""
#     COPY (SELECT * FROM pm.price_history) TO 'price_history.parquet' (FORMAT parquet)
# """)

In [1]:
import pandas as pd
import numpy as np

In [2]:
# an alternative to panda's insert(args) since it throws an error if the column already exists
# same logic but instead of df.insert, we use insert_or_replace(df, args)
def insert_or_replace(df, position, name, values):
    if name in df.columns:
        df.drop(columns=name, inplace=True)
    df.insert(position, name, values)

# gives the position to insert columns after a specified column
from typing import cast

# for some reason pylance never lets me use .get_loc() + 1
def nextpos(df, col:str)->int:
    pos = cast(int, df.columns.get_loc(col))
    return pos + 1

# if i ever need to swap cols
def swapcolumns(df, col1:str, col2:str):
    columns = list(df.columns)
    new = columns.index(col1) 
    old = columns.index(col2)

    columns[new], columns[old] = columns[old], columns[new]

    df = df[columns]
    return df

# we want to convert the string format outcomes column into numeric values
import json
def extract_outcomes(outcome_prices_str):
    prices = json.loads(outcome_prices_str) # json lets python iterate over it

    # all outcomes are stored as [yes, no]
    # so getting the first index gets the string value of yes
    yes_price_str = prices[0] 

    yes_outcome = float(yes_price_str)
    return yes_outcome


In [11]:
df = pd.read_parquet("price_history.parquet")

# fidelity is always 60 mins so why even keep it?
df = df.drop(columns=["fidelity_min"])

# convert to category dtype in attempt to save memory
for col in ['token_id', 'market_id']:
    df[col] = df[col].astype('category')

# strangely 3 negative p values and 107 above 1, so we'll clip them
df["p"] = df["p"].clip(0, 1)

# use smaller datatypes
df["t"] = df["t"].astype("uint32")
df["p"] = df["p"].astype("float32")

### Before optimizations

| Column | Data type | Memory (MB) |
|:-------|:----------|------------:|
| `token_id` | `str` | 3690.913 |
| `market_id` | `str` | 617.620 |
| `t` | `int64` | 347.186 |
| `p` | `float64` | 347.186 |
| `fidelity_min` | `int64` | 347.186 |
| **Total** | — | **5350.091** |

### After optimizations

| Column | Data type | Memory (MB) |
|:-------|:----------|------------:|
| `token_id` | `category` | 177.825 |
| `market_id` | `category` | 174.318 |
| `t` | `uint32` | 173.593 |
| `p` | `float32` | 173.593 |
| **Total** | — | **699.329** |

### Overall improvement

| Metric | Before | After | Improvement |
|:-------|--------:|-------:|------------:|
| Total memory | 5350.091 MB | 699.329 MB | −4650.762 MB |
| Reduction | — | — | **86.9%** |
| Relative size | 100% | 13.1% | **7.65× smaller** |

In [4]:
df2 = pd.read_parquet("finalsample.parquet")

In [5]:
finalsample = df2[["id", "question", "endDate", "outcomes", "outcomePrices", "volumeNum", "createdAt", "closedTime", "clobTokenIds", "negRisk", "umaEndDate", "umaResolutionStatus", "eventStartTime", "gameStartTime"]].copy()
finalsample.head()

,id,question,endDate,outcomes,outcomePrices,volumeNum,createdAt,closedTime,clobTokenIds,negRisk,umaEndDate,umaResolutionStatus,eventStartTime,gameStartTime
0,253591,Will Donald Trump win the 2024 US Presidential...,2024-11-05T12:00:00Z,"[""Yes"", ""No""]","[""1"", ""0""]",1.531479e+09,2024-01-04T17:33:51.332Z,2024-11-06 15:17:41+00,"[""21742633143463906290569050155826241533067272...",True,2024-11-06T15:17:41Z,resolved,NaN,NaN
1,253597,Will Kamala Harris win the 2024 US Presidentia...,2024-11-04T12:00:00Z,"[""Yes"", ""No""]","[""0"", ""1""]",1.037039e+09,2024-01-04T17:40:17.792Z,2024-11-06 18:03:54+00,"[""69236923620077691027083946871148646972011131...",True,2024-11-06T18:03:54Z,resolved,NaN,NaN
2,511754,Will Donald Trump be inaugurated?,2025-01-20T12:00:00Z,"[""Yes"", ""No""]","[""1"", ""0""]",4.004095e+08,2024-11-01T20:59:58.040376Z,2025-01-20 22:51:36+00,"[""45714870634090908403813747458214625542376052...",True,2025-01-20T22:51:36Z,resolved,NaN,NaN
3,507892,Will the Sacramento Kings win the 2025 NBA Fin...,2025-06-23T12:00:00Z,"[""Yes"", ""No""]","[""0"", ""1""]",3.780115e+08,2024-09-24T15:33:30.214289Z,2025-04-17 08:51:19+00,"[""41257670817876317355997233596113195194194608...",True,2025-04-17T08:51:19Z,resolved,NaN,NaN
4,2169995,"MicroStrategy sells any Bitcoin by May 31, 2026?",2026-07-01T04:00:00Z,"[""Yes"", ""No""]","[""0"", ""1""]",3.758131e+08,2026-05-05T23:45:21.015988Z,2026-06-04 00:34:19+00,"[""25714007960293389110960044475283546872601238...",False,2026-06-04T00:34:19Z,resolved,NaN,2026-06-01 17:00:00+00


In [16]:
df2_no_negrisk = pd.read_parquet("finalsample_no_negrisk.parquet")

In [17]:
df2_negrisk = pd.read_parquet("finalsample_negrisk.parquet")

In [18]:
finalsample_no_negrisk = df2_no_negrisk[["id", "question", "endDate", "outcomes", "outcomePrices", "volumeNum", "createdAt", "closedTime", "clobTokenIds", "negRisk", "umaEndDate", "umaResolutionStatus", "eventStartTime", "gameStartTime"]].copy()

In [19]:
finalsample_negrisk = df2_negrisk[["id", "question", "endDate", "outcomes", "outcomePrices", "volumeNum", "createdAt", "closedTime", "clobTokenIds", "negRisk", "umaEndDate", "umaResolutionStatus", "eventStartTime", "gameStartTime"]].copy()

In [ ]:
finalsample_negrisk.negRisk.value_counts()

In [7]:
# equally splitting time bins doesnt make much sense, so these are 
# "meaningful" intervals
time_edges = [0, 24, 48, 96, 168, 336, 720, 1440, 2880, np.inf]

# so that we can read the meaningful intervals instead of just
# guessing from the total hours
time_labels = [
    "0-1 day",
    "1-2 days",
    "2-4 days",
    "4-7 days",
    "1-2 weeks",
    "2-4 weeks",
    "1-2 months",
    "2-4 months",
    ">4 months"
]

In [8]:
# combine all the cells into one function so i can make a panel
# for any df easily
def panel_construction_pipeline(sample, price_df,
                                time_edges = [0, 24, 48, 96, 168, 336, 720, 1440, 2880, np.inf],
                                time_labels = [
                                                "0-1 day",
                                                "1-2 days",
                                                "2-4 days",
                                                "4-7 days",
                                                "1-2 weeks",
                                                "2-4 weeks",
                                                "1-2 months",
                                                "2-4 months",
                                                ">4 months"
                                            ]):
    # we want to get the yes outcomes from clob and merge it
    # with the finalsamples df from gamma
    yes_outcomes_list = []

    # iterrows() lets you loop over the df one row at a time
    for index, row in sample.iterrows():
        outcome_prices = row["outcomePrices"]
        yes_outcomes = extract_outcomes(outcome_prices)
        yes_outcomes_list.append((yes_outcomes))

    sample["yes_outcomes"] = yes_outcomes_list


    # 14 entries with no volume
    # 3 entries with no closedTime
    sample = sample.dropna(subset=["volumeNum", "closedTime"])


    # apparently 12 markets dont have price 0 or 1, so we drop them
    sample = sample[(sample["yes_outcomes"] == 1.0) | (sample["yes_outcomes"] == 0.0)].copy()
    sample["yes_outcomes"] = sample["yes_outcomes"].astype("bool")


    # delete the redundant columns and leave just the 
    # bool outcome
    columns = list(sample.columns)
    new = columns.index("yes_outcomes") 
    old = columns.index("outcomes")

    columns[new], columns[old] = columns[old], columns[new]

    sample = sample[columns]
    sample = sample.drop(columns=["outcomes", "outcomePrices"])


    # storing just true or false as an object was taking 1.79mb
    # down to 0.05 after bool
    sample['negRisk'] = sample['negRisk'].astype(bool)


    # the time is stored as a string
    # so we'll convert it to datetime and then category
    # since that'll save mem in the panel

    # actually since we need to bin, we'll convert to category after that
    sample["closedTime"] = pd.to_datetime(sample["closedTime"], format="ISO8601", utc=True)
    sample["createdAt"] = pd.to_datetime(sample["createdAt"], format="ISO8601", utc=True)


    # the main problem with memory before was likely that clobtoken
    # and question were huge text fields with massive mem footprints
    # (8.5 and 3mb in a 50k row finalsample alone) that got duplicated
    # many many times in the 43.4m row final panel. so we'll only merge
    # tiny fields and keep finalsample 50k row with us as a ref
    # if needed later
    sample_for_merge = sample[['id', 'yes_outcomes', 'createdAt', 'closedTime', 'negRisk']].copy()


    # construct the panel dataset now
    # using inner bcz some ids in df dont exist in final sample
    # maybe cause of us dropping a few observations
    panel = price_df.merge(sample_for_merge, left_on = "market_id", right_on = "id", how = "inner")

    # create a MultiIndex of (market, time)
    # discard the old index and then create the new 2d index once we've sorted
    panel = panel.sort_values(["market_id", "t"]).reset_index(drop=True)
    panel = panel.set_index(["market_id", "t"])
    panel = panel.drop(columns=["id"]) # its the index now anyways


    # add time to resolution column
    time = panel.index.get_level_values("t")
    current_time = pd.to_datetime(time, unit="s", utc=True)

    # ttr in hours will be more useful later for calculations
    # even though its less readable
    ttr = (panel["closedTime"]-current_time).dt.total_seconds() / 3600

    # you can see the 1hr gaps between entries due to the 60 min fidelity
    insert_or_replace(panel, nextpos(panel, "closedTime"), "time_to_resolution", ttr)
    panel["time_to_resolution"] = panel["time_to_resolution"].astype("float32")

    # bin ttr into appropriate intervals defined above
    ttrbins = pd.cut(
        panel["time_to_resolution"],
        bins=time_edges,
        labels=time_labels,
        include_lowest=True,
        ordered=True # ensures that groupings respect the given order and doesnt go alphabetically or anything
    )

    insert_or_replace(panel, nextpos(panel, "time_to_resolution"), "ttr_bin", ttrbins)


    # bin prices
    edges = np.linspace(0, 1, 21)
    labels = [f"{edges[i]:.2f}-{edges[i+1]:.2f}" for i in range(len(edges)-1)] # avoids parquet conversion errors

    # no idea why pylance cant understand this syntax works
    # even after i tried explicitly declaring p as a series
    pb = pd.cut(panel["p"], bins=edges, labels=labels, include_lowest=True,ordered=True) # type: ignore[call-overload]

    # pb = pd.cut(panel["p"], bins=np.linspace(0,1,21), include_lowest=True) # type: ignore[call-overload]
    insert_or_replace(panel, nextpos(panel, "p"), "price_bins", pb)

    return panel

In [12]:
panel = panel_construction_pipeline(finalsample, df)

In [ ]:
panel.head()

In [20]:
panel_no_negrisk = panel_construction_pipeline(finalsample_no_negrisk, df)

In [21]:
panel_negrisk = panel_construction_pipeline(finalsample_negrisk, df)

In [ ]:
panel.memory_usage(deep=True) / 1e6 # MB per column

In [ ]:
# zstd is a better compression alg than the default snappy for this panel
# panel.to_parquet("panel.parquet", compression="zstd", index = True)

In [ ]:
panel_no_negrisk.to_parquet("panel_no_negrisk.parquet", compression="zstd", index = True)

In [ ]:
panel_negrisk.to_parquet("panel_negrisk.parquet", compression="zstd", index = True)

In [ ]:
# finalsample.to_parquet("companion_info.parquet", compression="zstd", index = True)

In [ ]:
# comprehensive summary stats maker
def summarize_columns(df, cols, dollar_cols=None):
    # explicitly define columns measured in dollars so we can return values
    # in millions of dollars
    if dollar_cols is None:
        dollar_cols = []

    rows = []
    for col in cols:
        series = df[col]

        if col in dollar_cols:
            series = series / 1e6 

        stat_row = {"n": series.count(), "missing": series.isna().sum(), "n_unique": series.nunique()}

        # handling diff types separately since not all summary stats
        # make sense for all dtypes
        # std dev for dates makes no sense
        if pd.api.types.is_bool_dtype(series):
            stat_row["mean"] = series.mean() 
            stat_row["std"] = np.nan
            stat_row["min"] = np.nan
            stat_row["median"] = np.nan
            stat_row["max"] = np.nan
        elif pd.api.types.is_datetime64_any_dtype(series):
            stat_row["mean"] = series.mean()
            stat_row["std"] = np.nan
            stat_row["min"] = series.min()
            stat_row["median"] = series.median()
            stat_row["max"] = series.max()
        elif pd.api.types.is_numeric_dtype(series):
            stat_row["mean"] = series.mean()
            stat_row["std"] = series.std()
            stat_row["min"] = series.min()
            stat_row["median"] = series.median()
            stat_row["max"] = series.max()
        else:
            for key in ["mean", "std", "min", "median", "max"]: # just in case
                stat_row[key] = np.nan

        label = col + "_millions" if col in dollar_cols else col
        stat_row["column"] = label
        rows.append(stat_row)

    summary = pd.DataFrame(rows).set_index("column")
    column_order = ["n", "missing", "n_unique", "mean", "median", "std", "min", "max"]
    return summary[column_order]

In [ ]:
# duration a market stayed open, in days
finalsample["duration_days"] = (finalsample["closedTime"] - finalsample["createdAt"]).dt.total_seconds() / 86400

market_cols = ["volumeNum", "duration_days", "negRisk", "yes_outcomes", "createdAt", "closedTime"]
market_summary = summarize_columns(finalsample, market_cols, dollar_cols=["volumeNum"])
market_summary.round(3) # summary of the 50k markets data

In [ ]:
# stats in days are more legible and it matches the market_days stat
panel["time_to_resolution_days"] = panel["time_to_resolution"] / 24
panel_cols = ["p", "time_to_resolution_days"]
panel_summary = summarize_columns(panel, panel_cols)
panel_summary.round(3) # panel summary

In [ ]:
# equal-weighted: average TTR within each market first, then average across markets
market_avg_ttr = panel.groupby("market_id", observed=True)["time_to_resolution"].mean() / 24  # to days
market_avg_ttr.describe()

In [ ]:
# treat the binned vars separately
ttr_bin_counts = panel["ttr_bin"].value_counts(sort=False).to_frame("n_rows")
ttr_bin_counts["pct_rows"] = (ttr_bin_counts["n_rows"] / len(panel) * 100).round(2)
ttr_bin_counts

In [ ]:
price_bin_counts = panel["price_bins"].value_counts(sort=False).to_frame("n_rows")
price_bin_counts["pct_rows"] = (price_bin_counts["n_rows"] / len(panel) * 100).round(2)
price_bin_counts

In [13]:
finalsample.groupby("negRisk")["yes_outcomes"].mean()

negRisk
False    0.368954
True     0.235668
Name: yes_outcomes, dtype: float64

In [22]:
finalsample_negrisk["yes_outcomes"].mean()

np.float64(0.23026995000000003)

In [23]:
finalsample_no_negrisk["yes_outcomes"].mean()

np.float64(0.39012)